# mAb Downstream Purification Process

This notebook demonstrates the bio manufacturing unit operations in difflow for a monoclonal antibody (mAb) purification process.

**Process Overview:**
1. Harvest clarification (centrifuge)
2. Protein A affinity capture
3. Ion exchange polishing
4. Ultrafiltration/diafiltration (UF/DF)

All operations are differentiable, enabling gradient-based optimization of process parameters.

In [1]:
import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)

from difflow import Flowsheet, Unit, make_stream, get_flows
from difflow_bio import (
    # Centrifuge
    DiscStackCentrifuge, DiscStackParams,
    # Chromatography
    ProteinAChromatography, ProteinAParams,
    IonExchangeChromatography, IEXParams,
    # Filtration
    Ultrafiltration, UltrafiltrationParams,
    Diafiltration, DiafiltrationParams,
)

## 1. Define the Process Species

A typical mAb harvest contains:
- **mAb**: The target product
- **cells**: CHO cells to be removed
- **HCP**: Host cell proteins (impurity)
- **DNA**: Host cell DNA (impurity)
- **buffer**: Buffer/water

In [2]:
species = ["mAb", "cells", "HCP", "DNA", "buffer"]

# Harvest from bioreactor (typical fed-batch harvest)
# Concentrations in g/L, flows represent total mass (g) for a 2000L batch
harvest_volume = 2000.0  # L
harvest = make_stream(
    {
        "mAb": 5.0 * harvest_volume,      # 5 g/L titer = 10 kg total
        "cells": 20.0 * harvest_volume,    # 20 g/L wet cell weight
        "HCP": 2.0 * harvest_volume,       # Host cell proteins
        "DNA": 0.1 * harvest_volume,       # DNA
        "buffer": 973.0 * harvest_volume,  # Water/buffer (balance)
    },
    T=310.0,  # 37C
    P=101325.0,
)

print("Harvest composition:")
for name, flow in get_flows(harvest).items():
    conc = float(flow) / harvest_volume
    print(f"  {name}: {conc:.2f} g/L ({float(flow)/1000:.1f} kg total)")

Harvest composition:
  mAb: 5.00 g/L (10.0 kg total)
  cells: 20.00 g/L (40.0 kg total)
  HCP: 2.00 g/L (4.0 kg total)
  DNA: 0.10 g/L (0.2 kg total)
  buffer: 973.00 g/L (1946.0 kg total)


## 2. Harvest Clarification with Disc-Stack Centrifuge

Remove cells from the harvest using a disc-stack centrifuge. Key parameters:
- Number of discs and geometry determine the Sigma factor (equivalent settling area)
- Higher Sigma = better separation but larger equipment
- Cell diameter and density affect settling velocity

In [3]:
# Configure disc-stack centrifuge
centrifuge_params = DiscStackParams(
    n_discs=100,
    r_outer=0.15,           # 15 cm outer radius
    r_inner=0.05,           # 5 cm inner radius
    half_angle=0.698,       # 40 degrees
    rpm=6000.0,
    efficiency=0.8,
    species_order=species,
    cell_species="cells",
)
centrifuge = DiscStackCentrifuge(centrifuge_params)

# Run centrifuge
# Q = volumetric flow rate in m³/s (or L/h depending on consistent units)
# Processing 2000 L over ~2 hours = ~1000 L/h = 2.78e-4 m³/s
Q = 2.78e-4  # m³/s
(concentrate, clarified), cent_info = centrifuge(
    harvest,
    Q=Q,
    d_particle=15e-6,     # CHO cells ~15 μm diameter
    rho_particle=1050.0,  # Cell density kg/m³
    rho_fluid=1000.0,     # Broth density ~water
    viscosity=0.001,      # ~water viscosity Pa·s
    concentrate_fraction=0.05,  # 5% goes to concentrate
)

print("Centrifuge Results:")
print(f"  Sigma factor: {float(centrifuge.sigma):.0f} m²")
print(f"  Cell recovery to concentrate: {float(cent_info['cell_recovery'])*100:.1f}%")
print(f"  Critical particle diameter: {float(cent_info['critical_diameter'])*1e6:.1f} μm")

# Calculate mAb recovery in clarified stream
mab_in = float(get_flows(harvest)["mAb"])
mab_clarified = float(get_flows(clarified)["mAb"])
print(f"  mAb in clarified: {mab_clarified/mab_in*100:.1f}%")

print(f"\nClarified stream (per species):")
for name, flow in get_flows(clarified).items():
    print(f"  {name}: {float(flow):.1f} g")

Centrifuge Results:
  Sigma factor: 32654 m²
  Cell recovery to concentrate: 80.0%
  Critical particle diameter: 0.2 μm
  mAb in clarified: 95.0%

Clarified stream (per species):
  mAb: 9500.0 g
  cells: 8000.0 g
  HCP: 3800.0 g
  DNA: 190.0 g
  buffer: 1848700.0 g


## 3. Protein A Affinity Capture

Protein A chromatography is the workhorse of mAb purification:
- Highly selective binding to Fc region of IgG
- Typical binding capacity: 30-50 g/L resin
- Achieves 2-3 log HCP clearance

In [4]:
# Configure Protein A column
proa_params = ProteinAParams(
    column_volume=25.0,     # 25 L column for 10 kg mAb
    q_max=40.0,             # 40 g/L binding capacity
    K_d=0.05,               # High affinity
    target_species="mAb",
    yield_factor=0.95,      # 95% elution yield
    impurity_clearance={
        "HCP": 2.0,         # 100x reduction
        "DNA": 3.0,         # 1000x reduction  
        "cells": 4.0,       # Complete removal
    },
    species_order=species,
)
proa = ProteinAChromatography(proa_params)

# Run capture step
# Load volume = clarified harvest volume (~95% of original)
load_volume = harvest_volume * 0.95
(proa_eluate, proa_waste), proa_info = proa(
    clarified,
    load_volume=load_volume,
    breakthrough_limit=0.01,  # 1% breakthrough
)

print("Protein A Results:")
print(f"  mAb yield: {float(proa_info['yield'])*100:.1f}%")
print(f"  Purity: {float(proa_info['purity'])*100:.1f}%")
print(f"  DBC used: {float(proa_info['DBC']):.1f} g/L")

print(f"\nEluate composition:")
for name, flow in get_flows(proa_eluate).items():
    if float(flow) > 0.01:
        print(f"  {name}: {float(flow):.2f} g")

Protein A Results:
  mAb yield: 95.0%
  Purity: 0.5%
  DBC used: 39.6 g/L

Eluate composition:
  mAb: 9.17 g
  HCP: 0.04 g
  buffer: 1878.17 g


## 4. Ion Exchange Polishing (CEX)

Cation exchange chromatography provides additional purification:
- Removes aggregates and charge variants
- Can operate in bind-elute or flow-through mode

In [5]:
# Configure CEX column (bind-elute mode)
cex_params = IEXParams(
    column_volume=10.0,
    mode="bind_elute",
    q_max=60.0,
    K_d=0.3,
    target_species="mAb",
    selectivity={
        "mAb": 1.0,      # Binds strongly
        "HCP": 0.3,      # Some binding
        "DNA": 0.0,      # Does not bind (flows through)
        "cells": 0.0,
        "buffer": 0.0,
    },
    yield_factor=0.92,
    species_order=species,
)
cex = IonExchangeChromatography(cex_params)

# Run polishing step (elution pool volume ~5 CV)
cex_load_volume = 5.0 * proa_params.column_volume  # Elution pool volume
(cex_product, cex_waste), cex_info = cex(proa_eluate, load_volume=cex_load_volume)

print("CEX Results:")
print(f"  mAb yield: {float(cex_info['yield'])*100:.1f}%")
print(f"  Purity: {float(cex_info['purity'])*100:.1f}%")

print(f"\nProduct composition:")
for name, flow in get_flows(cex_product).items():
    if float(flow) > 0.001:
        print(f"  {name}: {float(flow):.3f} g")

CEX Results:
  mAb yield: 92.0%
  Purity: 4.3%

Product composition:
  mAb: 0.559 g
  buffer: 12.439 g


## 5. UF/DF for Final Formulation

Ultrafiltration concentrates the product, then diafiltration exchanges into final formulation buffer.

In [6]:
# Ultrafiltration - concentrate 10x
uf_params = UltrafiltrationParams(
    membrane_area=2.0,  # m²
    MWCO=30.0,          # 30 kDa cutoff
    rejection={"mAb": 0.999, "HCP": 0.95, "DNA": 0.0},  # mAb fully retained
    species_order=species,
)
uf = Ultrafiltration(uf_params)

(uf_retentate, uf_permeate), uf_info = uf(cex_product, concentration_factor=10.0)

# Get mAb recovery
mab_cex = float(get_flows(cex_product).get("mAb", 0))
mab_uf = float(get_flows(uf_retentate).get("mAb", 0))
uf_recovery = mab_uf / mab_cex if mab_cex > 0 else 0

print("UF Results:")
print(f"  mAb recovery: {uf_recovery*100:.1f}%")
print(f"  Concentration factor: {float(uf_info['concentration_factor']):.1f}x")
print(f"  Volume reduction: {float(uf_info['volume_reduction'])*100:.1f}%")

UF Results:
  mAb recovery: 99.9%
  Concentration factor: 10.0x
  Volume reduction: 90.0%


In [7]:
# Diafiltration - 5 diavolumes for buffer exchange
df_params = DiafiltrationParams(
    membrane_area=2.0,
    MWCO=30.0,
    rejection={"mAb": 0.999, "HCP": 0.90, "DNA": 0.0},
    species_order=species,
)
df = Diafiltration(df_params)

# Add formulation buffer
formulation_buffer = make_stream(
    {"mAb": 0.0, "cells": 0.0, "HCP": 0.0, "DNA": 0.0, "buffer": 1000.0},
    T=298.0,
    P=101325.0,
)

(df_product, df_waste), df_info = df(uf_retentate, formulation_buffer, n_diavolumes=5.0)

# Calculate DF recovery
mab_df = float(get_flows(df_product).get("mAb", 0))
df_recovery = mab_df / mab_uf if mab_uf > 0 else 0

print("DF Results:")
print(f"  mAb recovery: {df_recovery*100:.1f}%")
print(f"  Diavolumes: {float(df_info.get('n_diavolumes', 5.0)):.1f}")

DF Results:
  mAb recovery: 99.5%
  Diavolumes: 5.0


## 6. Process Summary

In [8]:
# Calculate overall yields
harvest_mab = float(get_flows(harvest)["mAb"])
final_mab = float(get_flows(df_product).get("mAb", 0))
overall_yield = final_mab / harvest_mab if harvest_mab > 0 else 0

# Calculate purity
final_flows = get_flows(df_product)
final_mab_mass = float(final_flows.get("mAb", 0))
final_hcp_mass = float(final_flows.get("HCP", 0))
total_protein = final_mab_mass + final_hcp_mass
final_purity = final_mab_mass / total_protein if total_protein > 0 else 1.0

print("="*50)
print("PROCESS SUMMARY")
print("="*50)
print(f"\nStarting material: {harvest_mab/1000:.1f} kg mAb")
print(f"Final product: {final_mab/1000:.2f} kg mAb")
print(f"\nOverall yield: {overall_yield*100:.1f}%")
print(f"Final purity: {final_purity*100:.2f}%")

# Step-by-step yields
mab_clarified = float(get_flows(clarified).get("mAb", 0))
mab_proa = float(get_flows(proa_eluate).get("mAb", 0))
mab_cex = float(get_flows(cex_product).get("mAb", 0))

print(f"\nStep yields:")
print(f"  Centrifuge: {mab_clarified/harvest_mab*100:.1f}%")
print(f"  Protein A: {mab_proa/mab_clarified*100:.1f}%" if mab_clarified > 0 else "  Protein A: N/A")
print(f"  CEX: {mab_cex/mab_proa*100:.1f}%" if mab_proa > 0 else "  CEX: N/A")
print(f"  UF: {uf_recovery*100:.1f}%")
print(f"  DF: {df_recovery*100:.1f}%")

PROCESS SUMMARY

Starting material: 10.0 kg mAb
Final product: 0.00 kg mAb

Overall yield: 0.0%
Final purity: 99.98%

Step yields:
  Centrifuge: 95.0%
  Protein A: 0.1%
  CEX: 6.1%
  UF: 99.9%
  DF: 99.5%


## 7. Differentiability: Sensitivity Analysis

Because all operations are JAX-differentiable, we can compute gradients of any output with respect to any parameter.

In [9]:
def compute_final_yield(proa_yield_factor):
    """Compute overall yield as a function of Protein A yield."""
    # Recreate ProA with different yield
    params = ProteinAParams(
        column_volume=25.0,
        q_max=40.0,
        K_d=0.05,
        target_species="mAb",
        yield_factor=proa_yield_factor,
        impurity_clearance={"HCP": 2.0, "DNA": 3.0, "cells": 4.0},
        species_order=species,
    )
    proa_unit = ProteinAChromatography(params)
    
    # Run downstream (simplified - just ProA and CEX)
    (eluate, _), _ = proa_unit(clarified, load_volume=load_volume)
    (cex_out, _), _ = cex(eluate, load_volume=cex_load_volume)
    
    # Return yield
    return get_flows(cex_out).get("mAb", jnp.array(0.0)) / harvest_mab

# Compute gradient
yield_sensitivity = jax.grad(compute_final_yield)(jnp.array(0.95))
print(f"Sensitivity: d(yield)/d(ProA_yield_factor) = {float(yield_sensitivity):.3f}")
print(f"\nInterpretation: A 1% improvement in ProA elution yield")
print(f"increases overall yield by {float(yield_sensitivity)*0.01*100:.2f} percentage points")

Sensitivity: d(yield)/d(ProA_yield_factor) = 0.000

Interpretation: A 1% improvement in ProA elution yield
increases overall yield by 0.00 percentage points


In [10]:
# Sensitivity to centrifuge efficiency
def yield_vs_centrifuge_efficiency(efficiency):
    """Compute yield as function of centrifuge efficiency."""
    params = DiscStackParams(
        n_discs=100,
        r_outer=0.15,
        r_inner=0.05,
        half_angle=0.698,
        rpm=6000.0,
        efficiency=efficiency,
        species_order=species,
        cell_species="cells",
    )
    cent = DiscStackCentrifuge(params)
    
    (_, clar), _ = cent(
        harvest, Q=Q, d_particle=15e-6,
        concentrate_fraction=0.05
    )
    
    return get_flows(clar).get("mAb", jnp.array(0.0)) / harvest_mab

cent_sensitivity = jax.grad(yield_vs_centrifuge_efficiency)(jnp.array(0.8))
print(f"Sensitivity: d(clarified_yield)/d(centrifuge_efficiency) = {float(cent_sensitivity):.3f}")

Sensitivity: d(clarified_yield)/d(centrifuge_efficiency) = 0.000


## 8. Visualize the Process

Use the visualization module to create an interactive flowsheet diagram.

In [11]:
try:
    from difflow.visualization import FlowsheetGraph, render_flowsheet
    
    # Build graph manually for this multi-output process
    graph = FlowsheetGraph(name="mAb Downstream Process")
    
    # Add nodes
    graph.add_node("centrifuge", "Disc-Stack Centrifuge", unit_type="DiscStackCentrifuge")
    graph.add_node("proa", "Protein A Capture", unit_type="ProteinAChromatography")
    graph.add_node("cex", "CEX Polish", unit_type="IonExchangeChromatography")
    graph.add_node("uf", "UF Concentrate", unit_type="Ultrafiltration")
    graph.add_node("df", "DF Buffer Exchange", unit_type="Diafiltration")
    
    # Add feed and connections
    graph.add_feed("centrifuge", "Harvest")
    graph.add_edge("centrifuge", "proa", edge_id="Clarified")
    graph.add_edge("proa", "cex", edge_id="Eluate")
    graph.add_edge("cex", "uf", edge_id="CEX Product")
    graph.add_edge("uf", "df", edge_id="Concentrated")
    graph.add_product("df", "Drug Substance")
    graph.add_product("centrifuge", "Cell Paste", source_port="cells")
    
    fig = render_flowsheet(graph, layout="spring", title="mAb Downstream Process")
    fig.show()
except ImportError:
    print("Install plotly and networkx for visualization:")
    print("  pip install plotly networkx")

## Summary

This notebook demonstrated:

1. **Bio manufacturing unit operations**: Centrifuge, chromatography, and filtration
2. **Realistic mAb purification**: Industry-standard process train
3. **Differentiability**: Gradient computation for sensitivity analysis
4. **Visualization**: Interactive process flow diagrams

All operations support automatic differentiation, enabling:
- Process optimization
- Uncertainty quantification
- Model-based control design